
# CAT3D-Wind clumpy torus: wind fraction and viewing angle

The CAT3D-Wind torus (Hönig & Kishimoto 2017) splits the circumnuclear dust
into a mid-plane clumpy disc plus a polar outflow ("wind"). Its infrared
reprocessing is controlled by three observables: the wind mass fraction
``fwd``, the radial cloud-distribution index ``a``, and the viewing angle
``cos i``.

This example sweeps the **wind fraction** (``fwd``, 1.0 -> 2.25) at two viewing
angles. A larger polar-wind component fills in the near/mid-IR and shifts the
balance of warm vs cool dust emission; the effect depends on whether the system
is viewed close to face-on (left) or edge-on (right), since the wind is
polar-directed. The torus contribution is isolated by subtracting the disc-only
SED (the torus normalizes to the disc luminosity).

## References
.. [1] S. F. Hönig & M. Kishimoto, "The dusty heart of nearby active galaxies.
   II. From clumpy torus models to a unified model," ApJL 838, L20 (2017).
   arXiv:1702.08691.
.. [2] L. N. Martínez-Ramírez et al., "AGNfitter-rx: Modeling the radio-to-X-ray
   spectral energy distributions of AGNs," A&A 688, A46 (2024).
   arXiv:2405.12111. https://doi.org/10.1051/0004-6361/202449329


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style
from tengri.utils.physics_constants import C_AA  # speed of light [Angstrom/s]

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

ssp = tengri.load_ssp()

SFH = {"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0}
DUST = {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

# CAT3D-Wind grid: fwd in [1.0, 2.25]. Two viewing angles (near face-on / edge-on).
FWD_VALUES = np.linspace(1.0, 2.25, 7)
COS_INC = {"face-on (cos i = 0.85)": 0.85, "edge-on (cos i = 0.2)": 0.2}

BASE_AGN = {
    "disc": {"type": "multicolor", "all_params": tengri.FIXED},
    "all_params": tengri.FIXED,
    "log_lbol": 12.0,
    "lum_ratio": 1.0,
}


def _build_sed(torus: dict | None) -> tuple[np.ndarray, np.ndarray]:
    agn = dict(BASE_AGN)
    if torus is not None:
        agn["torus"] = torus
    model = tengri.SEDModel.build(ssp, sfh=SFH, dust=DUST, agn=agn, redshift=tengri.Fixed(0.05))
    p = dict(model.spec.sample(jax.random.PRNGKey(0)))
    out = model.predict(p)
    return np.asarray(model.wavelengths), np.asarray(out.rest_sed())


WAVE, DISC_ONLY = _build_sed(None)


def torus_sed(cos_inc: float, fwd: float) -> tuple[np.ndarray, np.ndarray]:
    """Return (wavelength [AA], nu*L_nu [erg/s]) for the CAT3D-Wind torus alone."""
    wave, total = _build_sed(
        {"type": "cat3d_wind", "all_params": tengri.FIXED, "cos_inc": cos_inc, "fwd_cat3d": fwd}
    )
    disc = np.interp(wave, WAVE, DISC_ONLY)
    return wave, C_AA / wave * np.clip(total - disc, 0.0, None)


fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.6), sharey=True)
norm = mpl.colors.Normalize(vmin=FWD_VALUES.min(), vmax=FWD_VALUES.max())
cmap = plt.get_cmap("viridis")

for ax, (label, cos_inc) in zip(axes, COS_INC.items()):
    for fwd in FWD_VALUES:
        wave, nu_l_nu = torus_sed(cos_inc, fwd)
        ax.loglog(wave, nu_l_nu, color=cmap(norm(fwd)), lw=1.6)
    ax.set_xlim(3e3, 3e6)
    ax.set_ylim(3e43, 3e45)
    ax.set_xlabel(r"Rest-frame wavelength $\lambda$  [$\mathrm{\AA}$]")
    ax.set_title(label, fontsize=10)
    ax.grid(True, which="major", alpha=0.2)

axes[0].set_ylabel(r"$\nu L_\nu$  [erg s$^{-1}$]")
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
cb = fig.colorbar(sm, ax=axes, pad=0.01, fraction=0.046)
cb.set_label(r"wind mass fraction  $f_\mathrm{wd}$", fontsize=9)

fig.suptitle(
    "CAT3D-Wind torus: wind fraction and viewing angle",
    fontsize=11.5,
    weight="bold",
)
plt.savefig("plot_cat3d_wind_sweep.png", dpi=150, bbox_inches="tight")